# Master PDF Builder

**Cell 2** — Converts Copilot JSON sessions from `Copilot_inbox\` into PDFs, drops them into `inbox\`  
**Cell 1** — Merges everything in `inbox\` (PDFs, screenshots, Copilot exports) into `master.pdf`

### One-time setup
```
pip install pypdf reportlab Pillow
```

### Folder structure (auto-created)
```
C:\MasterPDF\
  Copilot_inbox\   ← Copy your Copilot .json session files here manually
  inbox\           ← Drop PDFs & screenshots here; Cell 2 also writes PDFs here
  done\            ← Processed files moved here automatically
  master.pdf       ← Your growing combined PDF
```

---

## How to export a Copilot chat from VS Code

Do this whenever you want to save a Copilot session:

1. In the Copilot chat panel in VS Code, right-click → **Copy All**
2. Open Notepad, paste, and save as a `.txt` file (e.g. `my_session.txt`)
3. Drop that file into `C:\MasterPDF\Copilot_inbox\`
4. Run **Cell 2** below — it parses the full conversation (your prompts + Copilot's answers), converts it to a formatted PDF, drops it into `inbox\`, then automatically runs Cell 1 to merge everything into `master.pdf`

> **Tip:** You can drop multiple `.txt` files into `Copilot_inbox\` at once — Cell 2 processes them all in one go.

---

## How to add other PDFs or screenshots

1. Drop any `.pdf`, `.png`, `.jpg` (etc.) files into `C:\MasterPDF\inbox\`
2. Run **Cell 1** directly — no need to run Cell 2 first

---

In [1]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Master PDF Builder                                ║
# ║  Merges all PDFs and images in inbox\ into master.pdf       ║
# ╚══════════════════════════════════════════════════════════════╝

import os, sys, shutil, datetime, io
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────
BASE_DIR   = Path(r"C:\Users\Cleveland\Desktop\MasterPDF")
INBOX_DIR  = BASE_DIR / "inbox"
DONE_DIR   = BASE_DIR / "done"
MASTER_PDF = BASE_DIR / "master.pdf"

SUPPORTED_PDF   = {".pdf"}
SUPPORTED_IMAGE = {".png", ".jpg", ".jpeg", ".bmp", ".gif", ".tiff", ".tif", ".webp"}
# ─────────────────────────────────────────────────────────────────


def check_dependencies():
    missing = []
    try:
        import pypdf
    except ImportError:
        missing.append("pypdf")
    try:
        from reportlab.pdfgen import canvas
    except ImportError:
        missing.append("reportlab")
    try:
        from PIL import Image
    except ImportError:
        missing.append("Pillow")
    if missing:
        print(f"❌ Missing packages — run: pip install {' '.join(missing)}")
        return False
    return True


def setup_folders():
    for folder in [INBOX_DIR, DONE_DIR]:
        folder.mkdir(parents=True, exist_ok=True)
    print(f"📁 Folders ready under: {BASE_DIR}")


def image_to_pdf_bytes(image_path: Path) -> bytes:
    from PIL import Image
    from reportlab.pdfgen import canvas
    from reportlab.lib.pagesizes import A4

    img = Image.open(image_path)
    if img.mode in ("RGBA", "P"):
        img = img.convert("RGB")

    page_w, page_h = A4
    margin = 20
    max_w, max_h = page_w - 2 * margin, page_h - 2 * margin
    img_w, img_h = img.size
    scale = min(max_w / img_w, max_h / img_h, 1.0)
    draw_w, draw_h = img_w * scale, img_h * scale
    x = (page_w - draw_w) / 2
    y = (page_h - draw_h) / 2

    pdf_buffer = io.BytesIO()
    c = canvas.Canvas(pdf_buffer, pagesize=A4)
    c.drawImage(image_path.as_posix(), x, y, width=draw_w, height=draw_h, preserveAspectRatio=True)
    c.save()
    pdf_buffer.seek(0)
    return pdf_buffer.read()


def collect_new_files():
    files = []
    for f in sorted(INBOX_DIR.iterdir()):
        if not f.is_file():
            continue
        ext = f.suffix.lower()
        if ext in SUPPORTED_PDF or ext in SUPPORTED_IMAGE:
            files.append(f)
    return files


def append_to_master(new_files):
    from pypdf import PdfWriter, PdfReader

    writer = PdfWriter()
    if MASTER_PDF.exists():
        existing = PdfReader(str(MASTER_PDF))
        for page in existing.pages:
            writer.add_page(page)
        print(f"📖 Loaded existing master.pdf ({len(existing.pages)} pages)")
    else:
        print("🆕 No master.pdf yet — creating fresh.")

    added_pages = 0
    processed_files = []

    for f in new_files:
        ext = f.suffix.lower()
        try:
            if ext in SUPPORTED_PDF:
                reader = PdfReader(str(f))
                for page in reader.pages:
                    writer.add_page(page)
                n = len(reader.pages)
                print(f"  ✅ PDF:   {f.name}  ({n} page{'s' if n != 1 else ''})")
                added_pages += n
            elif ext in SUPPORTED_IMAGE:
                pdf_bytes = image_to_pdf_bytes(f)
                reader = PdfReader(io.BytesIO(pdf_bytes))
                for page in reader.pages:
                    writer.add_page(page)
                print(f"  ✅ Image: {f.name}  (1 page)")
                added_pages += 1
            processed_files.append(f)
        except Exception as e:
            print(f"  ⚠️  Skipped {f.name}: {e}")

    if added_pages == 0:
        print("ℹ️  Nothing new was added.")
        return

    with open(MASTER_PDF, "wb") as out:
        writer.write(out)

    total_pages = len(PdfReader(str(MASTER_PDF)).pages)
    print(f"\n✅ master.pdf updated — {added_pages} page(s) added, {total_pages} total pages.")

    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    done_session = DONE_DIR / timestamp
    done_session.mkdir(parents=True, exist_ok=True)
    for f in processed_files:
        dest = done_session / f.name
        if dest.exists():
            dest = done_session / f"{f.stem}_{timestamp}{f.suffix}"
        shutil.move(str(f), str(dest))
    print(f"📦 Processed files moved to: done\\{timestamp}\\")


# ── Run ──────────────────────────────────────────────────────────
print("=" * 55)
print("  CELL 1 — Master PDF Builder")
print("=" * 55)

if check_dependencies():
    setup_folders()
    new_files = collect_new_files()
    if not new_files:
        print(f"\nℹ️  Inbox is empty. Drop files into:\n   {INBOX_DIR}")
    else:
        print(f"\n📥 Found {len(new_files)} file(s) in inbox:")
        for f in new_files:
            print(f"   - {f.name}")
        print()
        append_to_master(new_files)
        print(f"\n📄 Master PDF location: {MASTER_PDF}")

  CELL 1 — Master PDF Builder
📁 Folders ready under: C:\Users\Cleveland\Desktop\MasterPDF

📥 Found 1 file(s) in inbox:
   - Implementing dimensionality reduction algorithms in Python - Claude.pdf

🆕 No master.pdf yet — creating fresh.
  ✅ PDF:   Implementing dimensionality reduction algorithms in Python - Claude.pdf  (4 pages)

✅ master.pdf updated — 4 page(s) added, 4 total pages.
📦 Processed files moved to: done\20260609_121837\

📄 Master PDF location: C:\Users\Cleveland\Desktop\MasterPDF\master.pdf


In [2]:
# ╔══════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Copilot Chat → PDF                                ║
# ║  Converts .txt exports from Copilot_inbox\ to formatted     ║
# ║  PDFs and drops them into inbox\ ready for Cell 1           ║
# ╚══════════════════════════════════════════════════════════════╝
#
# Workflow:
#   1. In VS Code, right-click in the Copilot chat panel → Copy All
#   2. Paste into Notepad and save as a .txt file
#   3. Drop the .txt file into:  C:\MasterPDF\Copilot_inbox\
#   4. Run this cell — converts to PDF, drops into inbox\, then runs Cell 1

import re, datetime, shutil
from pathlib import Path

# ── Configuration ────────────────────────────────────────────────
COPILOT_INBOX = BASE_DIR / "Copilot_inbox"
COPILOT_DONE  = BASE_DIR / "done" / "copilot_txt"
# ─────────────────────────────────────────────────────────────────


def find_copilot_txts():
    """Return all .txt files in Copilot_inbox\."""
    if not COPILOT_INBOX.exists():
        COPILOT_INBOX.mkdir(parents=True, exist_ok=True)
        print(f"📁 Created folder: {COPILOT_INBOX}")
        print(f"   Paste your Copilot .txt exports there, then re-run.")
        return []
    files = sorted(COPILOT_INBOX.glob("*.txt"))
    return files


def parse_txt_session(txt_path: Path):
    """
    Parse a plain-text Copilot export with the format:
        User: <prompt text>
        GitHub Copilot: <response text>
        User: ...
        GitHub Copilot: ...

    Returns a list of {role, text} dicts, or None if unreadable/empty.
    """
    try:
        raw = txt_path.read_text(encoding="utf-8", errors="replace")
    except Exception as e:
        print(f"   ⚠️  Could not read {txt_path.name}: {e}")
        return None

    messages = []

    # Split on the speaker labels — keeps the label with its block
    # Matches "User:" or "GitHub Copilot:" at the start of a line
    pattern = re.compile(r"^(User:|GitHub Copilot:)", re.MULTILINE)
    parts = pattern.split(raw)

    # parts is: ['', 'User:', ' text...', 'GitHub Copilot:', ' text...', ...]
    # Walk in pairs of (label, content)
    i = 1
    while i < len(parts) - 1:
        label = parts[i].strip()
        content = parts[i + 1].strip()
        if label == "User:":
            role = "user"
        elif label == "GitHub Copilot:":
            role = "assistant"
        else:
            i += 2
            continue
        if content:
            messages.append({"role": role, "text": content})
        i += 2

    return messages if messages else None


def txt_session_to_pdf(messages, session_name: str, out_path: Path):
    """Render a list of chat messages as a formatted PDF."""
    from reportlab.lib.pagesizes import A4
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.lib import colors
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, HRFlowable
    from reportlab.lib.units import mm

    doc = SimpleDocTemplate(
        str(out_path),
        pagesize=A4,
        leftMargin=20*mm, rightMargin=20*mm,
        topMargin=20*mm, bottomMargin=20*mm
    )

    styles = getSampleStyleSheet()

    style_title = ParagraphStyle("Title2", parent=styles["Heading1"],
                                  fontSize=14, spaceAfter=4)
    style_meta  = ParagraphStyle("Meta",   parent=styles["Normal"],
                                  fontSize=8, textColor=colors.grey, spaceAfter=10)
    style_user  = ParagraphStyle("User",   parent=styles["Normal"],
                                  fontSize=10, leading=14, spaceAfter=4,
                                  textColor=colors.HexColor("#1a3c6e"),
                                  fontName="Helvetica-Bold")
    style_asst  = ParagraphStyle("Asst",   parent=styles["Normal"],
                                  fontSize=10, leading=14, spaceAfter=4,
                                  textColor=colors.HexColor("#2e4a1e"),
                                  fontName="Helvetica-Bold")
    style_body  = ParagraphStyle("Body",   parent=styles["Normal"],
                                  fontSize=9, leading=13, spaceAfter=6,
                                  fontName="Helvetica")
    style_code  = ParagraphStyle("Code",   parent=styles["Normal"],
                                  fontSize=8, leading=11, spaceAfter=2,
                                  fontName="Courier",
                                  backColor=colors.HexColor("#f4f4f4"),
                                  leftIndent=10)

    story = []

    # Header
    story.append(Paragraph("GitHub Copilot Chat Session", style_title))
    story.append(Paragraph(
        f"File: {session_name} &nbsp;|&nbsp; "
        f"Exported: {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}",
        style_meta
    ))
    story.append(HRFlowable(width="100%", thickness=1, color=colors.lightgrey, spaceAfter=12))

    for msg in messages:
        role = msg["role"]
        text = msg["text"]

        if role == "user":
            story.append(Paragraph("🧑 You", style_user))
        else:
            story.append(Paragraph("🤖 Copilot", style_asst))

        # Split into code blocks vs plain text
        parts = re.split(r"(```[\s\S]*?```)", text)
        for part in parts:
            if part.startswith("```"):
                code_text = re.sub(r"```\w*\n?", "", part).strip()
                for line in code_text.splitlines():
                    safe = line.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
                    story.append(Paragraph(safe or " ", style_code))
                story.append(Spacer(1, 4))
            else:
                for para in part.split("\n\n"):
                    para = para.strip()
                    if not para:
                        continue
                    # Single newlines become spaces for flow text
                    para = para.replace("\n", " ")
                    safe = para.replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
                    story.append(Paragraph(safe, style_body))

        story.append(HRFlowable(width="100%", thickness=0.5,
                                color=colors.HexColor("#e0e0e0"),
                                spaceBefore=4, spaceAfter=10))

    doc.build(story)


# ── Run ──────────────────────────────────────────────────────────
print("=" * 55)
print("  CELL 2 — Copilot Chat .txt → PDF")
print("=" * 55)

BASE_DIR.mkdir(parents=True, exist_ok=True)
INBOX_DIR.mkdir(parents=True, exist_ok=True)
COPILOT_DONE.mkdir(parents=True, exist_ok=True)

txt_files = find_copilot_txts()

if not txt_files:
    print(f"\nℹ️  No .txt files found in:\n   {COPILOT_INBOX}")
    print("   Copy All from Copilot, paste into Notepad, save as .txt, drop it there.")
else:
    print(f"\n📥 Found {len(txt_files)} file(s) in Copilot_inbox:")
    for f in txt_files:
        print(f"   - {f.name}")
    print()

    converted = 0
    for txt_path in txt_files:
        messages = parse_txt_session(txt_path)
        if not messages:
            print(f"   ⚠️  Skipped (empty/unreadable): {txt_path.name}")
            shutil.move(str(txt_path), str(COPILOT_DONE / txt_path.name))
            continue

        ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
        safe_name = re.sub(r"[^\w\-]", "_", txt_path.stem)[:40]
        out_pdf = INBOX_DIR / f"copilot_{ts}_{safe_name}.pdf"

        try:
            txt_session_to_pdf(messages, txt_path.stem, out_pdf)
            print(f"   ✅ {txt_path.name} → {out_pdf.name}  ({len(messages)} messages)")
            shutil.move(str(txt_path), str(COPILOT_DONE / txt_path.name))
            converted += 1
        except Exception as e:
            print(f"   ❌ Failed: {txt_path.name}: {e}")

    print(f"\n✅ {converted} file(s) converted and dropped into inbox\\")

    # ── Automatically run Cell 1 ──────────────────────────────────
    if converted > 0:
        print("\n" + "=" * 55)
        print("  Handing off to Cell 1...")
        print("=" * 55 + "\n")
        new_files = collect_new_files()
        if new_files:
            print(f"📥 Found {len(new_files)} file(s) in inbox:")
            for f in new_files:
                print(f"   - {f.name}")
            print()
            append_to_master(new_files)
            print(f"\n📄 Master PDF location: {MASTER_PDF}")
        else:
            print("ℹ️  inbox\\ is empty — nothing to merge into master.pdf.")

  CELL 2 — Copilot Chat .txt → PDF

📥 Found 1 file(s) in Copilot_inbox:
   - User Error Traceback (most recent c.txt

   ✅ User Error Traceback (most recent c.txt → copilot_20260609_121841_User_Error_Traceback__most_recent_c.pdf  (4 messages)

✅ 1 file(s) converted and dropped into inbox\

  Handing off to Cell 1...

📥 Found 1 file(s) in inbox:
   - copilot_20260609_121841_User_Error_Traceback__most_recent_c.pdf

📖 Loaded existing master.pdf (4 pages)
  ✅ PDF:   copilot_20260609_121841_User_Error_Traceback__most_recent_c.pdf  (4 pages)

✅ master.pdf updated — 4 page(s) added, 8 total pages.
📦 Processed files moved to: done\20260609_121841\

📄 Master PDF location: C:\Users\Cleveland\Desktop\MasterPDF\master.pdf
